# Fase 0: Introduzione


In questo Jupyter Notebook sarà possibile consultare la realizzazione del primo obiettivo del Project Work "Triage automatico dei ticket con Machine Learning". Gli obiettivi da raggiungere sono:
- *Creazione di un piccolo dataset sintetico (200–500 ticket testuali) con etichetta di categoria e priorità.*
    - Sintesi di ticket brevi (titolo + descrizione) con lessico tipico per Amministrazione, Tecnico, Commerciale.
    - Etichette di priorità semplici basate su parole chiave (es. 'errore', 'fattura', 'bloccante').
    - Esportazione CSV con colonne: id, title, body, category, priority.


Nello specifico, viene presentata un'analisi dei processi che mi hanno portato alla creazione di diversi dataset con diversa qualità. Il percorso è composto da 6 fasi:
- Fase 1: Caricamento e pulizia dei dati
    - Per la generazione del dataset tramite codice è fondamentale l'utilizzo della pseudo-casualità. Per comprendere meglio questa tecnica, ho deciso di iniziare dalla creazione di utenti realistici. Partendo da liste di nomi e cognomi reali, lo script genera combinazioni e relative email completamente inedite. Ho deciso di mantenere questo output nonostante non fosse richiesto dalla consegna per rendere il dataset più realistico. In questa fase, i dati in input vengono estratti e riordinati dai file "nomi_f.csv", "nomi_m.csv" e "cognomi.csv".
- Fase 2: Generazione pool di utenti (nome ed email)
    - In questa seconda fase viene illustrato il processo di generazione degli utenti come descritto precedentemente
- Fase 3: Generazione ticket brevi con lessico tipico casuale
    - Qui si entra nel core del progetto: ho sfruttato le meccaniche utilizzate nella fase 2 per creare un dataset elementare con lessico tipico casuale
- Fase 4: Esportazione e analisi del file dei ticket
    - In questa sezione è possibile vedere ed analizzare il risultato della fase 3. Come descritto nel capitolo, il dataset elementare creato nella fase precedente è valido, ma mostra ampi margini di miglioramento
- Fase 5: Ottimizzazione del dataset
    - Vengono implementate ed esaminate le ottimizzazioni e i correttivi discussi nella fase precedente.
- Fase 6: Generazione dataset avanzato tramite LLM (Gemma 4)
    - Qui ho sfruttato le potenzialità di Gemma 4 (un modello di LLM utilizzabbile in locale senza limiti di utilizzi) per creare un ulteriore dataset ancora più verosimile. L'intero codice è consultabile presso il [Notebook 2: Generazione Dataset avanzato con Gemma](02_Generazione_Dataset_Con_Gemma.ipynb)
- Fase 7: Analisi finale dei dataset
    - Un confronto diretto tra i diversi dataset generati lungo il percorso
 

_Librerie utilizzate nel progetto:_ 

In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Fase 1: Caricamento e pulizia dei dati

Come prima cosa, ho letto in input i file per i nomi e i cognomi, con lo scopo di creare delle email casuali. Tutti e tre i file sono stati scaricati sotto forma di txt al seguente link: https://github.com/Max1234-Ita/Liste. Ho preferito convertirli in formato csv per avere una struttura più ordinata e leggibile.

In [3]:
import pandas as pd
dfFemale = pd.read_csv(
    "nomi_f.csv", comment="#", header=None, names=["Nome", "Rarità"]
)
dfMale = pd.read_csv(
    "nomi_m.csv", comment="#", header=None, names=["Nome", "Rarità"]
)
df_cognomi = pd.read_csv("cognomi.csv", comment="#", header=None, names=["Cognome"])
df_nomi = pd.concat([dfFemale, dfMale], ignore_index=True)
df_nomi.sample(10)

,Nome,Rarità
25,ADEA,4
6666,ITO,4
249,ANASTASIA,2
1813,GASPARINA,3
8283,RODOSINDO,4
8788,UGONE,3
6538,IDELBERTO,4
1208,ETRUSCA,4
5693,ESQUILIO,4
5201,CORINZIO,4


A questo punto, ho a disposizione due diversi DataFrame:
- df_nomi: contiene una lista di nomi maschili e femminili insieme ad un indice che ne indica la sua rarità (1 diffuso, 4 raro)
- df_cognomi: contiene una lista con solo cognomi

# Fase 2: Generazione pool di utenti (nome ed email)

Partendo dai due DataFrame, posso adesso creare una lista di persone con nomi fittizzi e, di conseguenza, delle email fatte apposta per loro.
Il primo problema da affrontare è trovare un meccanismo per rendere i nomi più rari meno presenti all'interno della pool. Per fare questo, ho assegnato un peso probabilistico alle varie categorie. Stessa logica viene applicata anche ad una pool di domini email.

In [4]:
categorie_nomi = [1, 2, 3, 4]
pesi_categorie_nomi = [0.85, 0.11, 0.03, 0.01]
domini = ["gmail.com", "outlook.it", "hotmail.com", "libero.it", "yahoo.com"]
pesi_domini = [0.55, 0.15, 0.13, 0.13, 0.04]

Altra elemento frequente nelle email è la data di nascita dell'utente. Ho realizzato la funzione *genera_anno_realistico* che, usando la funzione *random.randint()* sorteggia un anno casuale. Lo userò quando creerò i vari utenti per decidere il loro anno di nascita.

In [5]:
import random
def genera_anno_realistico():
    anno = random.randint(1970, 2007)
    return str(anno)[2:]

Arriva finalmente la parte dove tutti i dati analizzati fino ad ora entrano in gioco. In questo ciclo, vengono estratti casualmente nomi, cognomi, domini per essere combinati dinamicamente seguendo diversi pattern strutturali; questo piccolo dettaglio rende le email meno statiche e più realistiche.
Da notare inoltre che, nell'estazione del nome, non vi è una totale casualità grazie al controllo dei pesi (*weights*).

A valle, troviamo la popolazione di una lista *pool_utenti* che conterrà *numero_utenti_unici* nomi inventati e *numero_utenti_unici* email corrispettive.

In [6]:
numero_utenti_unici = 125
pool_utenti = []
for _ in range(numero_utenti_unici):
    cat_nome = random.choices(categorie_nomi, weights=pesi_categorie_nomi)[0]
    df_nomi_filtrato = df_nomi[df_nomi["Rarità"] == cat_nome]
    if df_nomi_filtrato.empty:
        df_nomi_filtrato = df_nomi

    # Conserviamo una versione "pulita" con le iniziali maiuscole
    nome_reale = df_nomi_filtrato["Nome"].sample(n=1).values[0].title().strip()
    cognome_reale = df_cognomi["Cognome"].sample(n=1).values[0].title().strip()
    nome_completo = f"{nome_reale} {cognome_reale}"

    # Versione minuscola e senza spazi per l'indirizzo email
    nome_email = nome_reale.lower().replace(" ", "").replace("'", "")
    cognome_email = cognome_reale.lower().replace(" ", "").replace("'", "")

    dominio = random.choices(domini, weights=pesi_domini)[0]
    anno_nascita = genera_anno_realistico()

    formati = [
        f"{nome_email}.{cognome_email}",
        f"{nome_email}.{cognome_email}{anno_nascita}",
        f"{nome_email[0]}.{cognome_email}",
        f"{nome_email[0]}.{cognome_email}{anno_nascita}",
        f"{cognome_email}.{nome_email}",
        f"{nome_email}_{cognome_email}",
        f"{nome_email}{cognome_email}{anno_nascita}",
    ]
    pesi_formati = [0.20, 0.40, 0.10, 0.10, 0.10, 0.05, 0.05]
    struttura_email = random.choices(formati, weights=pesi_formati)[0]
    email_finale = f"{struttura_email}@{dominio}"

    pool_utenti.append({"user_name": nome_completo, "user_email": email_finale})

# Fase 3: Generazione ticket brevi con lessico tipico casuale

A questo punto, sarà necessario creare il nucleo centrale del progetto: la generazione vera e propria dei ticket.
Per farlo, ho creato un file JSON contenente una dizionario lessicale (che può facilmente essere esteso o adattato a piacimento). Vedi *lessico.json*.

Il JSON contiene una divisione per 3 tipologie di ticket, come da consegna:
- Amministrazione
- Tecnico
- Commerciale

Ogni divisione contiene a sua volte 3 liste, ovvero:
- Verbi
- Sostantivi
- Dettagli del ticket (frasi o contesti specifici)

Procedo dunque nel leggere il JSON e nel creare un metodo che, prendendo in input il nome del JSON, il pool di utenti, il numero di ticket da generare e il nome del file in output. Lo script è in grado di creare *num_ticket* tickets, estraendo senza rimozione un utente casuale di volta in volta.

In maniera del tutto casuale, lo script seleziona una categoria ed estra un verbo, un sostantivo e dei dettagli da combinare in un body che formerà una frase. 

Inizialmente avevo deciso di implementatare una logica di assegnazione delle priorità dove molto banalmente si assegnava una priorità più alta o bassa in base alla presenza o meno di determinate parole chiave. Ho deciso di migliorarla considerando una lista di keywords con priorità alta o media e di contare le loro ricorrenze all'interno del body del ticket.

A valle, il metodo organizza questo insieme di dati estratti casualmente in un unico record dove troviamo:
- id: valore incrementale che funge da chiave primaria univoca
- user_name: nome e cognome di un utente estratto casualmente dalla pool creata nella fase 2
- user_email: indirizzo email associato al medesimo utente
- title: oggetto del ticket
- body: il corpo effettivo del ticket
- category: categoria del ticket (Amministrazione, Tecnico o Commerciale)
- priority: il livello di urgenza

In [7]:
import json
import random
import pandas as pd


def genera_ticket_sintetici_con_utenti(nome_file_input, pool_utenti, num_ticket=200,nome_file_output="ticket_utenti_simulati.csv"):
    with open(nome_file_input, "r", encoding="utf-8") as f:
        lessico = json.load(f)

    categorie = list(lessico.keys())
    # Keywords più importanti
    keywords_alta = {
        "bloccante",
        "offline",
        "crash",
        "urgente",
        "aiuto",
        "leak",
        "picco di traffico",
        "scaduto",
        "sospendere",
        "penale",
    }
    keywords_media = {
        "errore",
        "errato",
        "sollecitare",
        "sollecito",
        "lento",
        "bug",
        "mancante",
        "disdire",
        "rettificare",
        "stornare",
        "problema",
    }

    pesi_oggetti = [0.20,0.25,0.10,0.15,0.05,0.10,0.10,0.05]
    pesi_formati = [0.15,0.20,0.15,0.10,0.05,0.10,0.10,0.11,0.01,0.01,0.01,0.01,0.01]

    utenti_estratti = random.choices(pool_utenti, k=num_ticket)

    dataset = []

    for i in range(num_ticket):
        utente = utenti_estratti[i]
        cat = random.choice(categorie)

        # Estrazione casuale delle componenti testuali
        v = random.choice(lessico[cat]["verbi"])
        s = random.choice(lessico[cat]["sostantivi"])
        d = random.choice(lessico[cat]["dettagli"])

        # Template per gli oggetti
        oggetti = [
            f"{v.capitalize()} {s}",
            f"URGENTE: {v.capitalize()} {s}",
            f"Richiesta supporto - categoria: {cat}",
            f"Segnalazione {s}",
            f"AIUTO - problema con {s}",
            f"Ticket {v.capitalize()} {s}",
            f"Richiesta info {s} {d}",
            f"{s} - {d}",
        ]
        title = random.choices(oggetti, weights=pesi_oggetti)[0]

        # Template per i corpi del testo
        formati = [
            f"Buongiorno,\ncon la presente si richiede cortesemente di {v} {s} {d}.\nRestiamo in attesa di un vostro riscontro urgente.\n\nCordiali saluti.",
            f"Ciao, per favore dovremmo {v} {s} {d} il prima possibile.\nFammi sapere si ci sono problemi. Grazie!",
            f"Salve, si richiede di {v} {s} {d}. Restiamo in attesa di un riscontro urgente.",
            f"Potete {v} {s} {d}?\nGrazie, saluti.",
            f"[NOTIFICA AUTOMATICA]: Rilevata necessità di {v} {s} {d}. Si prega di non rispondere a questa email.",
            f"Gentili colleghi,\nvi ricontatto per sapere se ci sono novità in merito alla richiesta di {v} {s} {d}.\nRimango a disposizione.\n\nBuon lavoro.",
            f"Purtroppo si è reso necessario {v} {s} {d}.\nProcedere appena possibile, blocca il lavoro del team.\nGrazie.",
            f"ciao, bisogna {v} {s} {d} entro oggi se riusciamo... fatemi sapere, grazie",
            f"Per curiosità stavo spulciando {s}, ho scoperto che {v} {s} {d}, e questo è un problema!",
            "ho un problema, quando posso chiamarvi?",
            f"Salve, sono {utente['user_name']}. Ho scoperto che {v} {s}:{d}",
            f"{v} {s} {d}.",
            f"sono {utente['user_name']}, ho un problema critico, ne possiamo parlare il prima possibile?",
        ]
        body = random.choices(formati, weights=pesi_formati)[0]

        testo_completo = f"{title} {body}".lower()

        # Controlla se la keyword è presente come sottostringa nel testo
        conteggio_alta = sum(1 for kw in keywords_alta if kw in testo_completo)
        conteggio_media = sum(1 for kw in keywords_media if kw in testo_completo)

        if conteggio_alta >= 2 and conteggio_media <= 1:
            priority = "Alta"
        elif conteggio_media >= 2 and conteggio_alta <= 1:
            priority = "Media"
        elif conteggio_alta > conteggio_media:
            priority = "Alta"
        elif conteggio_media > 0:
            priority = "Media"
        else:
            priority = "Bassa"

        dataset.append(
            {
                "id": f"TKT-{i+1:04d}",
                "user_name": utente["user_name"],
                "user_email": utente["user_email"],
                "title": title,
                "body": body,
                "category": cat,
                "priority": priority,
            }
        )

    # Scrittura finale ottimizzata
    df = pd.DataFrame(dataset)
    df.to_csv(nome_file_output, index=False, encoding="utf-8-sig")

    print(
        f"Successo! Creato il file '{nome_file_output}' con {num_ticket} righe totali."
    )

# Fase 4: Esportazione e analisi del file dei ticket

Finalmente si può mandare in esecuzione il metodo creato nella fase 3. L'output sarà la creazione di un file contenente utenti e ticket generati in maniera pseudocasuale (vedi *"ticket_utenti_simulati.csv"*)

Se lo si desidera, è possibile mandare in esecuzione questa le celle di questa fase per creare un nuovo dataset.

In [8]:
genera_ticket_sintetici_con_utenti("lessico.json", pool_utenti, 500,"ticket_utenti_simulati.csv")

Successo! Creato il file 'ticket_utenti_simulati.csv' con 500 righe totali.


Stampo parte del contenuto del file per vederne il risultato finale:

In [9]:
df_anteprima = pd.read_csv("ticket_utenti_simulati.csv")
df_anteprima.head(5)["body"]

0    Salve, si richiede di ripristinare un errore 5...
1    Purtroppo si è reso necessario proporre un pre...
2    Purtroppo si è reso necessario riscontrare un ...
3    Gentili colleghi,\nvi ricontatto per sapere se...
4    archiviare il giustificativo di spesa per dupl...
Name: body, dtype: str

In [10]:
import pandas as pd

df_anteprima = pd.read_csv("ticket_utenti_simulati.csv")

# Questo ti mostra il numero esatto di ticket per ogni priorità
print(df_anteprima["priority"].value_counts())

priority
Alta     216
Media    174
Bassa    110
Name: count, dtype: int64


Guardando i risultati è evidente che la generazione pseudocasuale dia già un buon risultato, ma si può fare di meglio. Il rischio attuale è che utilizzare un dataset con una struttura così rigida con poche varianti porti un eventuale LLM a memorizzare passivamente combinazioni fisse di parole al posto di imparare a fare un triage basato sul significato del problema del ticket. Facciamo un esempio:
- Se è presente la keyword "bug bloccante", allora il modello potrebbe spprendere questo possa essere solamente un ticket tecnico, cosa che non è così nella vita reale!

Più avanti addestrerò il modello anche con il dataset così composto per verificare se il dubbio sia lecito, ma nel frattempo ho deciso di risolvere il problema introducendo dell'ambiguità nel testo, creando delle frasi che introducano una percentuale di ticket con un vocabolario misto e simulare l'errore umano inserendo rumore (typo, refusi nel testo e diverso registro linguistico).

A questo scopo, l'analisi finale si baserà sul confronto di 3 diversi modelli addestrati con diversi dataset:
- Dataset grezzo creato con lo script della fase 3
- Dataset ottimizzato creato con lo script a seguire descritto
- Dataset generato da un modello di Intelligenza Artificiale generativa (Gemini)

# Fase 5: Ottimizzazione del dataset

Per superare la rigidità del primo dataset (Fase 3) e generarne uno più realistico, lo script è stato profondamente ottimizzato nella Fase 5. 

Il primo problema è stato quello di generare del rumore (*Noise Injection*). Questo è stato fatto creando la funzione 'introduci_typo', che prendendo in input un testo, sceglie casualmente un typo (come una ripetizione di lettere o un omissione)

Il secondo problema affrontato è stata l'aggiunta di ambiguità semantica. Adesso c'è una probabilità del 15% che il ticket usi parole incrociate da altre categorie. Questo significa che un ticket di una categoria (ad esempio Tecnico) ha una piccola probabilità di contenere anche parole appartenenti ad altre categorie.

Infine, ho arricchito i vettori 'oggetti' e 'formati' inserendo espressioni con registri linguistici diversi (alcuni professionali, altri un pò meno).

In [11]:
import json
import random
import pandas as pd


def introduci_typo(testo, probabilita=0.01):
    if random.random() > probabilita:
        return testo

    parole = testo.split()
    if not parole:
        return testo

    # Sceglie una parola a caso su cui applicare il refuso
    idx_parola = random.randint(0, len(parole) - 1)
    parola = parole[idx_parola]

    if len(parola) > 3:
        modalita = random.choice(
            ["inversione", "omissione", "ripetizione", "case"]
        )

        if modalita == "inversione":
            # (es. "errore" -> "erroer")
            idx = random.randint(1, len(parola) - 2)
            parola = (
                parola[:idx] + parola[idx + 1] + parola[idx] + parola[idx + 2 :]
            )
        elif modalita == "omissione":
            # (es. "backup" -> "bakup")
            idx = random.randint(1, len(parola) - 1)
            parola = parola[:idx] + parola[idx + 1 :]
        elif modalita == "ripetizione":
            # (es. "server" -> "servver")
            idx = random.randint(0, len(parola) - 1)
            parola = parola[:idx] + parola[idx] + parola[idx:]
        elif modalita == "case":
            # (es. "SSL" -> "Ssl" o "errore" -> "eRRORE")
            parola = parola.swapcase()

    parole[idx_parola] = parola
    return " ".join(parole)


def genera_ticket_sintetici_con_utenti(nome_file_input, pool_utenti, num_ticket=200,nome_file_output="ticket_utenti_simulati_pro.csv"):
    with open(nome_file_input, "r", encoding="utf-8") as f:
        lessico = json.load(f)

    categorie = list(lessico.keys())

    keywords_alta = {
        "bloccante",
        "offline",
        "crash",
        "urgente",
        "aiuto",
        "leak",
        "picco di traffico",
        "scaduto",
        "sospendere",
        "penale",
    }
    keywords_media = {
        "errore",
        "errato",
        "sollecitare",
        "sollecito",
        "lento",
        "bug",
        "mancante",
        "disdire",
        "rettificare",
        "stornare",
        "problema",
    }

    # Distribuzione dei pesi dei template (leggermente aggiornati per i nuovi formati)
    pesi_oggetti = [0.18,0.22,0.10,0.12,0.05,0.10,0.10,0.05,0.08]
    pesi_formati = [0.12,0.15,0.12,0.08,0.05,0.08,0.08,0.08,0.02,0.02,0.02,0.02,0.02,0.07,0.07]

    utenti_estratti = random.choices(pool_utenti, k=num_ticket)
    dataset = []

    for i in range(num_ticket):
        utente = utenti_estratti[i]
        cat_reale = random.choice(categorie)

        # C'è una probabilità del 15% che il ticket usi parole incrociate da altre categorie
        cat_verbo = (
            cat_reale if random.random() > 0.15 else random.choice(categorie)
        )
        cat_sost = (
            cat_reale if random.random() > 0.15 else random.choice(categorie)
        )
        cat_dett = (
            cat_reale if random.random() > 0.15 else random.choice(categorie)
        )

        v = random.choice(lessico[cat_verbo]["verbi"])
        s = random.choice(lessico[cat_sost]["sostantivi"])
        d = random.choice(lessico[cat_dett]["dettagli"])

        oggetti = [
            f"{v.capitalize()} {s}",
            f"URGENTE: {v.capitalize()} {s}",
            f"Richiesta supporto - categoria: {cat_reale}",
            f"Segnalazione {s}",
            f"AIUTO - problema con {s}",
            f"Ticket {v.capitalize()} {s}",
            f"Richiesta info {s} {d}",
            f"{s} - {d}",
            f"problema urgente {s} non va",
        ]
        title = random.choices(oggetti, weights=pesi_oggetti)[0]

        formati = [
            f"Buongiorno,\ncon la presente si richiede cortesemente di {v} {s} {d}.\nRestiamo in attesa di un vostro riscontro urgente.\n\nCordiali saluti.",
            f"Ciao, per favore dovremmo {v} {s} {d} il prima possibile.\nFammi sapere si ci sono problemi. Grazie!",
            f"Salve, si richiede di {v} {s} {d}. Restiamo in attesa di un riscontro urgente.",
            f"Potete {v} {s} {d}?\nGrazie, saluti.",
            f"[NOTIFICA AUTOMATICA]: Rilevata necessità di {v} {s} {d}. Si prega di non rispondere a questa email.",
            f"Gentili colleghi,\nvi ricontatto per sapere se ci sono novità in merito alla richiesta di {v} {s} {d}.\nRimango a disposizione.\n\nBuon lavoro.",
            f"Purtroppo si è reso necessario {v} {s} {d}.\nProcedere appena possibile, blocca il lavoro del team.\nGrazie.",
            f"ciao, bisogna {v} {s} {d} entro oggi se riusciamo... fatemi sapere, grazie",
            f"Per curiosità stavo spulciando {s}, ho scoperto che {v} {s} {d}, e questo è un problema!",
            "ho un problema, quando posso chiamarvi?",
            f"Salve, sono {utente['user_name']}. Ho scoperto che {v} {s}:{d}",
            f"{v} {s} {d}.",
            f"sono {utente['user_name']}, ho un problema critico, ne possiamo parlare il prima possibile?",
            f"scusate ma qui non funziona piu nulla!!! dobbiamo assolutamente {v} {s} o si blocca tutto il reparto!! rispondetemi asap",
            f"Riferimento {s}. Potete verificare? è collegato a {d}. grazie anche da parte del capo.",
        ]
        body = random.choices(formati, weights=pesi_formati)[0]

        # Applichiamo i typo sia al titolo che al corpo
        title = introduci_typo(title, probabilita=0.10)
        body = introduci_typo(body, probabilita=0.20)

        testo_completo = f"{title} {body}".lower()

        conteggio_alta = sum(1 for kw in keywords_alta if kw in testo_completo)
        conteggio_media = sum(1 for kw in keywords_media if kw in testo_completo)

        if conteggio_alta >= 2 and conteggio_media <= 1:
            priority = "Alta"
        elif conteggio_media >= 2 and conteggio_alta <= 1:
            priority = "Media"
        elif conteggio_alta > conteggio_media:
            priority = "Alta"
        elif conteggio_media > 0:
            priority = "Media"
        else:
            priority = "Bassa"

        dataset.append(
            {
                "id": f"TKT-{i+1:04d}",
                "user_name": utente["user_name"],
                "user_email": utente["user_email"],
                "title": title,
                "body": body,
                "category": cat_reale,
                "priority": priority,
            }
        )

    # Scrittura finale ottimizzata
    df = pd.DataFrame(dataset)
    df.to_csv(nome_file_output, index=False, encoding="utf-8-sig")

    print(
        f"Successo! Creato il file {nome_file_output} con {num_ticket} righe totali."
    )

In [12]:
genera_ticket_sintetici_con_utenti("lessico.json", pool_utenti, 500,"ticket_utenti_simulati_pro.csv")
df_anteprima = pd.read_csv("ticket_utenti_simulati_pro.csv")
df_anteprima.head(5)["body"]

Successo! Creato il file ticket_utenti_simulati_pro.csv con 500 righe totali.


0    Ciao, per favore dovremmo restituire la nota d...
1    ciao, bisogna restituire la nota di credito pe...
2              ho un problema, quando posso chiamarvi?
3    Salve, si richiede di negoziare la licenza agg...
4    Gentili colleghi,\nvi ricontatto per sapere se...
Name: body, dtype: str

# Fase 6: Generazione dataset avanzato tramite LLM (Gemma 4)

Entrambi gli script basati su regole si comportano già abbastanza bene, ma non sono ancora soddisfatto dell'output finale. La pseudorandomizzazione che contradistingue gli script rende i ticket innaturali e, nella maggior parte dei casi, molto ripetitivi. Per superare tale limite, ho deciso dunque di sfruttare la potenza dei modelli generativi.

Invece di optare per una normale query all'interno di interfacce web commerciali (come Gemini o soluzioni simili), ho deciso di utilizzare uno strumento molto più versatile e professionale, ovvero Gemma 4. Questo è un modello linguistico di grandi dimensione (Large Language Model) sviluppato da Google che ho configurato ed eseguito localmente tramite la piattaforma Ollama. Questa architettura ha permesso di aggirare i vincoli legati al rate limiting e i costi di consumo tipici delle API basate su cloud (come ad esempio Gemini, anch'essa di Google), garantendo al contempo il controllo totale sui parametri di generazione (come ad esempio la formattazione nativa della risposta). 

Per consultare la generazione avanzata tramite LLM, vai al file [Notebook 2: Generazione Dati con Gemma](02_Generazione_Dataset_Con_Gemma.ipynb).

# Fase 7 - Analisi finale dei dataset

È ovviamente presente una EVA più profonda all'interno del secondo file Jupyter Notebook.

In [13]:
genera_ticket_sintetici_con_utenti("lessico.json", pool_utenti, 500,"ticket_utenti_simulati_pro.csv")
df_anteprima = pd.read_csv("ticket_utenti_simulati.csv")
print("*** TICKET BASE ***")
df_anteprima.head(5)["body"]

Successo! Creato il file ticket_utenti_simulati_pro.csv con 500 righe totali.
*** TICKET BASE ***


0    Salve, si richiede di ripristinare un errore 5...
1    Purtroppo si è reso necessario proporre un pre...
2    Purtroppo si è reso necessario riscontrare un ...
3    Gentili colleghi,\nvi ricontatto per sapere se...
4    archiviare il giustificativo di spesa per dupl...
Name: body, dtype: str

In [14]:
df_anteprima = pd.read_csv("ticket_utenti_simulati_pro.csv")
print("*** TICKET MIGLIORATO ***")
df_anteprima.head(5)["body"]

*** TICKET MIGLIORATO ***


0    Salve, si richiede di configurare il picco di ...
1    Riferimento lo sconto commerciale. Potete veri...
2    Ciao, per favore dovremmo sospendere lo storno...
3    Salve, si richiede di testare un leak di memor...
4    Riferimento il giustificativo di spesa. Potete...
Name: body, dtype: str

# Conclusioni

Avendo consolidato la pipeline di generazione, la base dati è ora pronta per la validazione sul campo. Nel prossimi Jupyter Notebook del progetto si procederà all'**addestramento comparativo dei modelli di Machine Learning**. 

L'obiettivo sarà testare l'efficacia dei vari dataset generati per valutare  quale approccio garantisca la miglior capacità di generalizzazione, accuratezza e resilienza nel triage automatico dei ticket.